[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.d_model = d_model
        self.num_heads = num_heads # num q
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        bs, seq, _ = x.size()

        Q = self.W_q(x) # bs, seq, #h * dk = dm
        K = self.W_k(x) # bs, seq, #kvh * dk
        V = self.W_v(x)

        # dim break up into individual heads
        Q = Q.reshape(bs, seq, self.num_heads, self.d_k)
        K = K.reshape(bs, seq, self.num_kv_heads, self.d_k)
        V = V.reshape(bs, seq, self.num_kv_heads, self.d_k)
        
        # repeat kv dim to match q's
        K = torch.repeat_interleave(K, self.num_heads // self.num_kv_heads, dim=2) # bs, seq, #h, dk
        V = torch.repeat_interleave(V, self.num_heads // self.num_kv_heads, dim=2)

        # transpose to compute, for each batch and head, the attention of each word in the seq to one another
        Q = Q.transpose(1, 2) # bs, #h, seq, dk
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores = Q @ K.mT / math.sqrt(self.d_k) # bs, #h, seq, seq
        att = torch.softmax(scores, dim=-1) @ V # bs, #h, seq, dk
        # reshape attention to regular size
        att = att.transpose(1, 2) # bs, seq, #h, dk
        att = att.reshape(bs, seq, -1) # bs, #h, dm
        return att

In [11]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [12]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.2ms)
  ✅ [2/5] nn.Linear with correct shapes (1.2ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.2ms)
  ✅ [4/5] KV heads are shared correctly (1.5ms)
  ✅ [5/5] Gradient flow (3.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (12.1ms total)
  Progress saved. Run status() to see your dashboard.

